In [ ]:
pip install lifelines

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

# ===== 1. Basis opschonen =====
df = df.copy()

df["fetch_timestamp"] = pd.to_datetime(df["fetch_timestamp"], errors="coerce")
df = df.dropna(subset=["fetch_timestamp", "title", "categoryTitle"])
df = df.sort_values("fetch_timestamp")

# uniek product binnen winkel
df["product_key"] = (
    df["title"].astype(str).str.strip()
    + " | " +
    df["brand"].astype(str).str.strip()
)

# ===== 2. Survival table bouwen =====
dataset_end = df["fetch_timestamp"].max()

lifetimes = (
    df.groupby(["categoryTitle", "product_key"], as_index=False)
      .agg(
          start=("fetch_timestamp", "min"),
          end=("fetch_timestamp", "max")
      )
)

# duur in uren
lifetimes["duration_hours"] = (
    lifetimes["end"] - lifetimes["start"]
).dt.total_seconds() / 3600

lifetimes = lifetimes[lifetimes["duration_hours"] > 0]

# censored als item nog bestaat op laatste meetmoment
lifetimes["event"] = (lifetimes["end"] < dataset_end).astype(int)

# ===== 3. Survival plots per categoryTitle =====
kmf = KaplanMeierFitter()

unique_categories = lifetimes["categoryTitle"].unique()

for cat in sorted(unique_categories):

    subset = lifetimes[lifetimes["categoryTitle"] == cat]

    if len(subset) < 5:   # voorkomt ruis bij mini-samples
        continue

    kmf.fit(
        durations=subset["duration_hours"],
        event_observed=subset["event"],
        label=cat
    )

    plt.figure(figsize=(8,5))
    kmf.plot_survival_function()

    plt.title(f"Survival curve — {cat} (n={len(subset)})")
    plt.xlabel("Tijd zichtbaar in Laatste Kans (uren)")
    plt.ylabel("Kans dat product nog beschikbaar is")
    plt.tight_layout()
    plt.show()

    print(
        f"{cat}: n={len(subset)}, "
        f"median survival = {kmf.median_survival_time_:.2f} uur"
    )
